In [1]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset
import pandas as pd
from PIL import Image
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.tree import DecisionTreeClassifier
import os

#Decision Tree
train_probs = pd.read_csv('/kaggle/input/train-tree/train_tree.csv')
train_labels = train_probs["label"]
train_probs = train_probs.iloc[:,1:-1]
train_probs = np.array(train_probs)
decision_tree = DecisionTreeClassifier(criterion='gini', 
                                       max_depth=8,
                                       min_samples_split=12)
decision_tree.fit(train_probs, train_labels)

def tf_transforms(image):
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0
    mean = [0.5, 0.5, 0.5]
    std = [0.5, 0.5, 0.5]
    image = (image - mean) / std
    image = np.expand_dims(image, axis=0)
    return image

torch_transforms = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#DenseNet
model1 = load_model("/kaggle/input/densenet/keras/default/1/DenseNet (1).keras")

#ResNet
model2 = torch.load("/kaggle/input/resnet50_70_512x512/pytorch/default/1/Resnet50_70_512x512.pth")
model2.to(device)


test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images"
test_images = os.listdir(test_dir)

combined_probs = []
image_ids = []
count = 0
length = len(test_images)
for test_image in test_images:
    image_ids.append(test_image)
    image = Image.open(f"{test_dir}/{test_image}").convert("RGB")
    image1 = tf_transforms(image)
    image2 = torch_transforms(image)
    image2 = image2.unsqueeze(0).to(device)

    test_prediction_1 = np.array(model1.predict(image1,verbose=False)[0])
    with torch.no_grad():
        test_prediction_2 = np.array(model2(image2).cpu()[0])
        del image

    combined_probs.append(np.concatenate((test_prediction_1,test_prediction_2)))
    count+=1
    print(f"Count:{count}/{length}", end="\r")

2026-04-28 05:50:11.253447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777355411.268653      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777355411.273271      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 05:50:11.293197: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/train-tree/train_tree.csv'

In [2]:
prediction = decision_tree.predict(combined_probs)

NameError: name 'decision_tree' is not defined

In [3]:
submission = pd.DataFrame({
    'image_id': image_ids,
    'label': prediction
})
submission.to_csv('submission.csv', index=False)

NameError: name 'image_ids' is not defined

In [4]:
submission

NameError: name 'submission' is not defined